# Step 2: Exploratory Data Analysis

## 1. Introduction

Step 1 confirmed the dataset is clean (0 missing values, 0 duplicates) and severely imbalanced (0.129% fraud). This notebook goes beyond that structural check to find **patterns that separate fraud from legitimate transactions**, and — critically — to check whether some of those patterns are genuine behavioral signal vs. artifacts of how PaySim simulates data (which would not generalize to a real system).

Everything below is computed directly from the full 6,362,620-row file. No numbers are assumed or invented. `step` is treated throughout as **simulated time** (1 step = 1 simulated hour, 744 steps ≈ 31 days) — not a real-world timestamp — per PaySim's documentation.

No model is trained in this notebook. Any derived columns created here (balance deltas, ratios, zero-balance flags) are **exploratory only**, used to inspect the data, and are not a finalized feature-engineering pipeline (that's Step 3).

## 2. Dataset loading

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_utils import load_raw_data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = load_raw_data()
df.shape

## 3. Transaction type analysis

**Why this matters:** transaction types have wildly different volumes. Looking only at raw fraud *counts* per type is misleading — a type with more fraud in absolute terms might actually be *safer per transaction* than a smaller type. Fraud *rate within type* is the number that should drive modeling decisions (e.g. whether to treat type as a strong gating feature).

In [ ]:
type_counts = df["type"].value_counts()
type_pct = (type_counts / len(df) * 100).round(4)
fraud_by_type = df.groupby("type")["isFraud"].sum()
fraud_rate_by_type = (df.groupby("type")["isFraud"].mean() * 100)

type_summary = pd.DataFrame({
    "count": type_counts,
    "pct_of_total": type_pct,
    "fraud_count": fraud_by_type,
    "fraud_rate_pct": fraud_rate_by_type.round(6),
}).sort_values("count", ascending=False)
type_summary

In [ ]:
order = type_counts.index

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(order, type_counts.values, color="#4C72B0")
ax.set_title("Transaction Volume by Type")
ax.set_xlabel("Transaction Type")
ax.set_ylabel("Number of Transactions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, v in enumerate(type_counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("../figures/eda/transaction_type_distribution.png", dpi=150)
plt.show()

In [ ]:
rate_sorted = fraud_rate_by_type.reindex(order)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(order, rate_sorted.values, color="#C44E52")
ax.set_title("Fraud Rate (%) by Transaction Type")
ax.set_xlabel("Transaction Type")
ax.set_ylabel("Fraud Rate (%)")
for i, v in enumerate(rate_sorted.values):
    ax.text(i, v, f"{v:.4f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("../figures/eda/fraud_rate_by_type.png", dpi=150)
plt.show()

**Finding:** fraud occurs *only* in TRANSFER and CASH_OUT (confirmed in Step 1). Within those two: TRANSFER has fraud rate **0.7688%**, CASH_OUT has **0.1840%** — TRANSFER is ~4x riskier per transaction, even though CASH_OUT has a slightly higher absolute fraud count (4,116 vs 4,097) because it has ~4x the volume. **Implication:** transaction type is a strong, legitimate feature (known at decision time), and PAYMENT/CASH_IN/DEBIT can plausibly be treated as a separate low-risk population.

## 4. Transaction amount analysis

**Why this matters:** if large amount alone separated fraud from legitimate transactions, we wouldn't need a model — a threshold rule would do. We check whether that's true. Amounts are heavily right-skewed, so we use log-scale views and do **not** remove outliers (a large amount is exactly the kind of transaction we most need to still be able to evaluate).

In [ ]:
df["amount"].describe()

In [ ]:
for q in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999, 1.0]:
    print(f"  {q:>6.3f}: {df['amount'].quantile(q):>15,.2f}")

In [ ]:
df.groupby("isFraud")["amount"].describe()

In [ ]:
legit_amt = df[df["isFraud"] == 0]["amount"]
fraud_amt = df[df["isFraud"] == 1]["amount"]

median_fraud = fraud_amt.median()
pct_legit_above_median_fraud = (legit_amt > median_fraud).mean() * 100
print(f"Fraud amount: min {fraud_amt.min():,.2f}, median {fraud_amt.median():,.2f}, max {fraud_amt.max():,.2f}")
print(f"Legit amount: min {legit_amt.min():,.2f}, median {legit_amt.median():,.2f}, max {legit_amt.max():,.2f}")
print(f"% of legit transactions exceeding the median fraud amount: {pct_legit_above_median_fraud:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(np.log1p(df["amount"]), bins=100, color="#4C72B0")
axes[0].set_title("log1p(amount) — All Transactions")
axes[0].set_xlabel("log1p(amount)")
axes[0].set_ylabel("Count")

axes[1].hist(np.log1p(legit_amt), bins=100, alpha=0.6, label="Legitimate", color="#4C72B0", density=True)
axes[1].hist(np.log1p(fraud_amt), bins=100, alpha=0.6, label="Fraud", color="#C44E52", density=True)
axes[1].set_title("log1p(amount) Density — Legit vs Fraud")
axes[1].set_xlabel("log1p(amount)")
axes[1].legend()
plt.tight_layout()
plt.savefig("../figures/eda/amount_distribution.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot([legit_amt, fraud_amt], tick_labels=["Legitimate", "Fraud"], showfliers=True)
ax.set_yscale("log")
ax.set_title("Transaction Amount by Class (log scale, outliers retained)")
ax.set_ylabel("Amount (log scale)")
plt.tight_layout()
plt.savefig("../figures/eda/fraud_amount_distribution.png", dpi=150)
plt.show()

**Finding:** fraud transactions do skew higher (median 441,423 vs. 74,685 for legit; mean 1,467,967 vs. 178,197), so amount has *some* signal (weak positive correlation with `isFraud`, see section 8). But **large amount alone is not sufficient to identify fraud**: the single largest transaction in the whole dataset (92,445,516.64, a TRANSFER) is legitimate, and 6.76% of *all legitimate* transactions exceed the median fraud amount — heavy overlap between the two distributions. A threshold-on-amount rule would generate far too many false positives.

**Also found:** 16 transactions have `amount == 0`, and every one of them is fraud (all `type == CASH_OUT`) — a tiny but perfectly separating edge case, discussed further in section 7.

## 5. Temporal analysis

**Why this matters:** `step` is simulated time (1 step = 1 hour, ~31 days total) — not a calendar timestamp. We check for concentration of fraud in specific windows and for volume drift, since this directly informs whether Step 4 should use a temporal split instead of a random one.

In [ ]:
print("step range:", df["step"].min(), "to", df["step"].max())
print("implied days:", df["step"].max() / 24)

# Exploratory-only temporal columns (not persisted, EDA use only)
day = (df["step"] - 1) // 24
hour_of_day = (df["step"] - 1) % 24

In [ ]:
daily = pd.DataFrame({"day": day, "isFraud": df["isFraud"]}).groupby("day").agg(
    txn_count=("isFraud", "size"), fraud_count=("isFraud", "sum")
)
daily["fraud_rate_pct"] = daily["fraud_count"] / daily["txn_count"] * 100
daily.describe()

In [ ]:
hourly = pd.DataFrame({"hour_of_day": hour_of_day, "isFraud": df["isFraud"]}).groupby("hour_of_day").agg(
    txn_count=("isFraud", "size"), fraud_count=("isFraud", "sum")
)
hourly["fraud_rate_pct"] = hourly["fraud_count"] / hourly["txn_count"] * 100
hourly

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(daily.index, daily["txn_count"], color="#4C72B0")
axes[0].set_title("Transaction Volume Over Time (by simulated day)")
axes[0].set_ylabel("Transaction Count")

axes[1].plot(daily.index, daily["fraud_count"], color="#C44E52")
axes[1].set_title("Fraud Count Over Time (by simulated day)")
axes[1].set_xlabel("Simulated Day")
axes[1].set_ylabel("Fraud Count")
plt.tight_layout()
plt.savefig("../figures/eda/transaction_volume_over_time.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(daily.index, daily["fraud_rate_pct"], color="#C44E52", marker="o", markersize=3)
axes[0].set_title("Fraud Rate (%) by Simulated Day")
axes[0].set_xlabel("Simulated Day")
axes[0].set_ylabel("Fraud Rate (%)")

axes[1].bar(hourly.index, hourly["txn_count"], color="#4C72B0", alpha=0.5, label="Txn volume")
ax2 = axes[1].twinx()
ax2.plot(hourly.index, hourly["fraud_rate_pct"], color="#C44E52", marker="o")
axes[1].set_title("Txn Volume (bars) vs Fraud Rate % (line) by Hour-of-Day")
axes[1].set_xlabel("Hour of Day (step mod 24)")
axes[1].set_ylabel("Transaction Volume")
ax2.set_ylabel("Fraud Rate (%)")
plt.tight_layout()
plt.savefig("../figures/eda/fraud_over_time.png", dpi=150)
plt.show()

**Finding (important, and a bit subtle):** the raw **fraud count per hour-of-day is roughly flat** (range ~300-370 across all 24 hours), but **legitimate transaction volume follows a strong daily human-activity cycle** — it collapses overnight (hour 3: only 1,241 transactions vs. hour 18: 647,814). Because fraud count stays flat while the denominator (legit volume) craters overnight, the **fraud rate spikes to 16-22% during hours 2-5** — not because fraud increases at night, but because there are far fewer legitimate transactions to divide by. A chart of "fraud rate over time" without also showing volume would be actively misleading.

Also notable: simulated day 2 has an anomalous volume drop (1,070 transactions vs. a typical day of ~400-500k) with fraud rate jumping to ~29% for that day — this looks like a simulation-generation artifact rather than a realistic pattern (documented in the notes as something to be aware of, not something to model as a real weekly cycle, since 31 days of data isn't enough to distinguish a true weekly effect from a one-off generation quirk).

**Implication for Step 4:** this genuine hour-of-day and day-level structure means a **random split would leak information** about the full temporal distribution into training; the split strategy should account for this (temporal split will be evaluated properly in Step 4).

## 6. Balance behavior

**Why this matters:** the balance columns are where PaySim's simulation mechanics show through most clearly. We build a few **exploratory-only** derived variables (not a final pipeline) to check whether balances behave differently for fraud vs. legitimate transactions, and whether that difference looks like real behavioral signal or a simulator artifact.

In [ ]:
# Exploratory-only variables (EDA use only, not a production feature pipeline)
orig_balance_delta = df["newbalanceOrig"] - df["oldbalanceOrg"]
dest_balance_delta = df["newbalanceDest"] - df["oldbalanceDest"]
amount_to_orig_balance_ratio = df["amount"] / (df["oldbalanceOrg"] + 1)
orig_balance_error = orig_balance_delta + df["amount"]   # ~0 if sender balance dropped by exactly `amount`
dest_balance_error = dest_balance_delta - df["amount"]   # ~0 if receiver balance rose by exactly `amount`

exp = pd.DataFrame({
    "isFraud": df["isFraud"],
    "type": df["type"],
    "orig_balance_error": orig_balance_error,
    "dest_balance_error": dest_balance_error,
    "amount_to_orig_balance_ratio": amount_to_orig_balance_ratio,
    "orig_zero_before": (df["oldbalanceOrg"] == 0),
    "orig_zero_after": (df["newbalanceOrig"] == 0),
    "dest_zero_before": (df["oldbalanceDest"] == 0),
    "dest_zero_after": (df["newbalanceDest"] == 0),
})

In [ ]:
print("% of rows where sender balance updates EXACTLY by the transaction amount (orig_balance_error == 0):")
print((exp["orig_balance_error"] == 0).groupby(exp["isFraud"]).mean() * 100)

print("\n% of rows where receiver balance updates EXACTLY by the transaction amount (dest_balance_error == 0):")
print((exp["dest_balance_error"] == 0).groupby(exp["isFraud"]).mean() * 100)

In [ ]:
sub = exp[exp["type"].isin(["TRANSFER", "CASH_OUT"])]
zero_flags = sub.groupby("isFraud")[["orig_zero_before", "orig_zero_after", "dest_zero_before", "dest_zero_after"]].mean() * 100
zero_flags

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sub_plot = exp[exp["type"].isin(["TRANSFER", "CASH_OUT"])]
axes[0].boxplot(
    [sub_plot[sub_plot["isFraud"] == 0]["orig_balance_error"].clip(-1e6, 1e6),
     sub_plot[sub_plot["isFraud"] == 1]["orig_balance_error"].clip(-1e6, 1e6)],
    tick_labels=["Legitimate", "Fraud"],
)
axes[0].set_title("Sender Balance Error\n(newOrig - oldOrg + amount), TRANSFER/CASH_OUT")
axes[0].set_ylabel("Balance error (clipped to +-1M)")

dest_zero_rate = sub.groupby("isFraud")[["dest_zero_before", "dest_zero_after"]].mean() * 100
dest_zero_rate.plot(kind="bar", ax=axes[1], color=["#4C72B0", "#C44E52"])
axes[1].set_title("Destination Zero-Balance Rate (%) by Class\nTRANSFER/CASH_OUT only")
axes[1].set_xlabel("isFraud")
axes[1].set_ylabel("% of transactions")
axes[1].set_xticklabels(["Legitimate", "Fraud"], rotation=0)
plt.tight_layout()
plt.savefig("../figures/eda/balance_behavior.png", dpi=150)
plt.show()

**Finding:** sender balances update *exactly* by the transaction amount for **99.16%** of fraud transactions vs. only **6.80%** of legitimate ones. Destination balance stays at exactly 0 both before and after for **49.81%** of fraud TRANSFER/CASH_OUT transactions vs. **0.45%** of legitimate ones. These are large, striking differences — investigated further as a likely leakage/artifact issue in section 7, not treated yet as confirmed real-world signal.

## 7. Suspicious-pattern analysis

**Why this matters:** the balance-error findings above hint at something stronger — a near-perfect discriminating pattern. If a simple rule catches almost all fraud with almost no false positives, that needs to be understood before modeling, because it likely reflects how PaySim *simulates* fraud (draining an account) rather than something transferable to a live system.

In [ ]:
# 'Fully drained sender account': old balance == amount, and new balance == 0
fully_drained = (df["oldbalanceOrg"] == df["amount"]) & (df["newbalanceOrig"] == 0) & (df["oldbalanceOrg"] > 0)
sub2 = df[df["type"].isin(["TRANSFER", "CASH_OUT"])]
fd_sub = fully_drained[sub2.index]

print("Rate of 'fully drained sender account' pattern by class (TRANSFER/CASH_OUT only):")
print(pd.crosstab(sub2["isFraud"], fd_sub, normalize="index") * 100)
print("\nRaw counts:")
print(pd.crosstab(sub2["isFraud"], fd_sub))

**This is the single strongest pattern found in the whole EDA:** among TRANSFER/CASH_OUT transactions, **0 out of 2,762,196 legitimate transactions** show the "fully drained sender account" pattern (`oldbalanceOrg == amount` and `newbalanceOrig == 0`), while **8,008 out of 8,213 fraud transactions (97.50%)** show it. This single rule alone would achieve near-perfect precision and ~97.5% recall on this dataset with zero real modeling. That's a red flag, not a triumph — it's almost certainly how PaySim's fraud-injection logic works internally (simulated fraud = drain the account), not a pattern a real fraudster is guaranteed to leave. Flagged clearly for Step 3/Step 11 discussion rather than treated as a free win.

In [ ]:
# Checked hypothesis: do fraud TRANSFERs route money through an account that is then used for a fraud CASH_OUT
# (a 'money mule' pattern, sometimes cited for PaySim)? Verified directly on IDs rather than assumed.
fraud_transfers = df[(df["isFraud"] == 1) & (df["type"] == "TRANSFER")]
fraud_cashouts = df[(df["isFraud"] == 1) & (df["type"] == "CASH_OUT")]

overlap = set(fraud_transfers["nameDest"]) & set(fraud_cashouts["nameOrig"])
print(f"Fraud TRANSFERs: {len(fraud_transfers)}, Fraud CASH_OUTs: {len(fraud_cashouts)}")
print(f"Accounts that are both a fraud-TRANSFER destination and a fraud-CASH_OUT origin: {len(overlap)}")

**Result: 0 overlapping accounts.** The commonly-cited "TRANSFER-then-CASH_OUT through the same mule account" narrative for PaySim does **not** hold up when checked directly against account IDs in this file — fraud TRANSFER and fraud CASH_OUT rows do not chain through shared `nameDest`/`nameOrig` IDs. Reporting this as a checked-and-rejected hypothesis rather than assuming it, since the goal here is accurate findings, not a good story.

## 8. Correlation analysis

**Why this matters:** correlation shows linear association only, and **association is not causation**. We check it mainly to (a) confirm expected accounting-identity relationships, and (b) see whether any raw column is suspiciously predictive on its own — while remembering that our strongest finding above (the fully-drained-account pattern) is a *conditional/nonlinear* rule, which plain correlation will *not* surface. That's an important lesson in itself: don't rely on correlation alone to judge a variable's importance for fraud.

In [ ]:
numeric_cols = ["step", "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest", "isFlaggedFraud", "isFraud"]
corr = df[numeric_cols].corr()
corr.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Matrix — Raw Numeric Columns")
plt.tight_layout()
plt.savefig("../figures/eda/correlation_matrix.png", dpi=150)
plt.show()

In [ ]:
corr["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)

**Findings:**
- `oldbalanceDest`/`newbalanceDest` correlate with `amount` (0.294 / 0.459) — **expected**, since a transaction's amount mechanically influences the recipient's balance change; this is an accounting identity, not a discovery.
- Every raw column's linear correlation with `isFraud` is weak: `amount` is the strongest at only **0.077**, followed by `isFlaggedFraud` (0.044, and excluded from modeling regardless — see section 9) and `step` (0.032). Balance columns are essentially uncorrelated (|r| < 0.011).
- **This does not contradict section 7.** It confirms that the near-perfect "fully drained account" signal is a **conditional relationship between multiple columns**, not something visible in a simple linear correlation with any single raw column. Feature engineering (Step 3) needs to construct that relationship explicitly (e.g. as a balance-difference or ratio feature) — the model won't get it for free from raw balances alone.

## 9. Leakage investigation

**Why this matters most:** a model that looks excellent in this dataset because of a simulation artifact would be a false success — it wouldn't reflect a deployable fraud-detection capability, and claiming that success in an interview would be inaccurate. Being explicit about this is a core part of doing this project honestly.

**Three-way breakdown of every raw column:**

| Column | Category | Reasoning |
|---|---|---|
| `step` | (1) Available at decision time | Known the instant a transaction is requested. |
| `type` | (1) Available at decision time | Known before the transaction is processed. |
| `amount` | (1) Available at decision time | The requested amount is known before approval. |
| `nameOrig`, `nameDest` | (1) Available at decision time | IDs are known upfront; not usable directly as features (near-unique), but usable for behavioral aggregation in Step 3 using only *prior* history. |
| `oldbalanceOrg`, `oldbalanceDest` | (1) Available at decision time | These are pre-transaction account states — legitimately known before the transaction is approved. |
| `newbalanceOrig`, `newbalanceDest` | (2) Only known after the transaction | These reflect the *result* of the transaction being applied. In a real system, you decide whether to approve a transaction *before* it changes the balance — so using the post-transaction balance as an input feature assumes information that wouldn't exist yet at decision time. They're safe to use only indirectly (e.g. to construct historical/behavioral features from *past, already-settled* transactions), never as "the current transaction's own outcome." |
| `isFlaggedFraud` | (2) / label-adjacent | A rule-based output, not a transaction attribute. Confirmed in Step 1 to catch only 16/8,213 fraud cases. **Excluded entirely as a model feature.** |
| `isFraud` | Target | The label being predicted. |

**Simulation artifacts that make the task unrealistically easy (category 3):**
1. **The "fully drained sender account" pattern** (section 7): 97.50% of fraud vs. 0% of legitimate TRANSFER/CASH_OUT transactions. This almost certainly reflects PaySim's internal fraud-generation rule (simulated fraud = take the entire balance) rather than a universal fraudster behavior. A real fraudster might withdraw a partial amount specifically to avoid this kind of detection.
2. **Destination zero-balance pattern**: 49.81% of fraud vs. 0.45% of legitimate TRANSFER/CASH_OUT transactions have `oldbalanceDest == newbalanceDest == 0`. Likely reflects how PaySim populates destination accounts for its simulated fraud recipients, not a general real-world tell.
3. **The 16 zero-amount transactions**, all fraud, all `CASH_OUT`, all with `oldbalanceDest == newbalanceDest` (unmoved) — an edge case that looks like a simulator quirk rather than a realistic transaction.

**Decision carried into Step 3:** `isFlaggedFraud` is dropped entirely. `newbalanceOrig`/`newbalanceDest` will only be used to construct *differences/ratios* (e.g. `oldbalanceOrg - newbalanceOrig` vs. `amount`), and any resulting near-perfect features (like the fully-drained-account flag) will be reported honestly as reflecting a dataset artifact, with that caveat carried through into the results and limitations sections rather than presented as a discovered fraud pattern.

## 10. Key findings (summary)

1. Fraud occurs exclusively in TRANSFER (0.7688% fraud rate) and CASH_OUT (0.1840% fraud rate) — 0% in PAYMENT/CASH_IN/DEBIT.
2. Fraud amounts skew higher (median 441,423 vs. 74,685) but overlap heavily with legitimate amounts — amount alone cannot separate the classes.
3. Fraud count per hour-of-day is roughly flat (~300-370); legitimate volume follows a strong daily cycle and collapses overnight — so "fraud rate over time" spikes overnight purely from a shrinking denominator, not more fraud. Always pair a rate chart with a volume chart.
4. The **"fully drained sender account"** pattern (`oldbalanceOrg == amount` and `newbalanceOrig == 0`) separates fraud from legitimate TRANSFER/CASH_OUT transactions almost perfectly (97.50% vs. 0%) — flagged as a likely simulation artifact, not a guaranteed real-world tell.
5. Destination-zero-balance rate is far higher for fraud (49.81%) than legitimate (0.45%) transactions in TRANSFER/CASH_OUT — same artifact caveat applies.
6. The "TRANSFER-then-CASH_OUT via a shared mule account" hypothesis was checked directly on account IDs and **does not hold** (0 overlapping accounts) — reported as a rejected hypothesis.
7. Raw linear correlations with `isFraud` are all weak (max 0.077, for `amount`) — the strongest real signal is a conditional/multi-column relationship, not visible from single-column correlation.
8. `isFlaggedFraud` and post-transaction balance columns (`newbalanceOrig`, `newbalanceDest`) carry leakage risk and are handled explicitly (excluded / used only as differences) rather than fed in raw.

## 11. Implications for feature engineering (Step 3 preview — not built here)

- Encode `type` (or at minimum a TRANSFER/CASH_OUT vs. other flag) as a feature — strongest single legitimate signal found.
- Engineer balance-difference/ratio features (e.g. `oldbalanceOrg - amount - newbalanceOrig`, `amount / (oldbalanceOrg + 1)`) rather than passing raw balances — this is how the strongest pattern found (section 7) gets exposed to a model, while keeping the underlying columns interpretable.
- Any behavioral feature built from `nameOrig`/`nameDest` history (transaction frequency, typical amount, etc.) must only use **prior** transactions relative to the current one — no using a customer's future transactions to describe their "normal" behavior at the current step.
- Carry `step`-derived features (hour-of-day) forward cautiously, and note the volume/rate confound from section 5 in any modeling narrative.
- Exclude `isFlaggedFraud` and raw `newbalanceOrig`/`newbalanceDest` (except as components of engineered differences) from the model's input feature set.
- Because fraud is concentrated in two types, consider whether restricting modeling/evaluation focus to TRANSFER/CASH_OUT (while still scoring the full dataset) is worth discussing in Step 4/Step 11.
- Report the strength of the fully-drained-account feature honestly in Step 11/limitations: high performance driven substantially by this feature should be labeled as *partly a reflection of the synthetic dataset's fraud-generation mechanism*, not purely a generalizable modeling achievement.